In [8]:
# --- FIX THE PYTHON PATH SO src/ CAN BE IMPORTED ---
from pathlib import Path
import sys
import os

# Your notebook is in ML/notebooks, so ML is one level up
PROJECT_ROOT = Path.cwd().parents[0]   # -> .../ML

print("Adding to sys.path:", PROJECT_ROOT)
# Add ML to sys.path at the FRONT
sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC PATH:", PROJECT_ROOT / "src")
print("sys.path[0]:", sys.path[0])


print("Notebook cwd:", os.getcwd())




# Debug: show last few search paths
print("\nsys.path (last entries):")
for p in sys.path[:5]:
    print(" -", p)

# Test import:
from src.config import PROCESSED_DIR, TRAIN_RACES
print("\nSUCCESS! PROCESSED_DIR =", PROCESSED_DIR)


Adding to sys.path: C:\Users\Josel\Dissertation\ML
CWD: C:\Users\Josel\Dissertation\ML\notebooks
PROJECT_ROOT: C:\Users\Josel\Dissertation\ML
SRC PATH: C:\Users\Josel\Dissertation\ML\src
sys.path[0]: C:\Users\Josel\Dissertation\ML
Notebook cwd: C:\Users\Josel\Dissertation\ML\notebooks

sys.path (last entries):
 - C:\Users\Josel\Dissertation\ML
 - C:\Users\Josel\Dissertation\ML
 - C:\Users\Josel\Dissertation\ML
 - C:\Users\Josel\anaconda3\envs\f1ai\python311.zip
 - C:\Users\Josel\anaconda3\envs\f1ai\DLLs

SUCCESS! PROCESSED_DIR = C:\Users\Josel\Dissertation\ML\data\processed


In [6]:
import fastf1
import pandas as pd
import numpy as np
import os

# Enable FastF1 cache (in project root or wherever you like)
cache_dir = PROJECT_ROOT / "fastf1_cache"
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(str(cache_dir))

cache_dir


WindowsPath('C:/Users/Josel/Dissertation/ML/fastf1_cache')

In [7]:
def make_lap_dataset_for_session(year: int, event: str, session_type: str = "R") -> pd.DataFrame:
    """
    Build a simple per-lap/per-driver dataset for a single race session.
    Returns a DataFrame with useful features and a basic pit label.
    """
    print(f"Loading {year} {event} {session_type}...")
    session = fastf1.get_session(year, event, session_type)
    session.load()
    laps = session.laps.copy()

    # Drop rows without driver code
    laps = laps[~laps["Driver"].isna()].copy()

    # Per-driver derived features: LapsSinceLastPit, StopsSoFar
    def add_driver_features(df):
        df = df.sort_values("LapNumber").copy()
        laps_since_pit = []
        counter = 0
        for _, row in df.iterrows():
            counter += 1
            laps_since_pit.append(counter)
            if pd.notna(row["PitInTime"]):
                counter = 0
        df["LapsSinceLastPit"] = laps_since_pit
        df["StopsSoFar"] = df["Stint"] - 1
        return df

    laps = laps.groupby("Driver", group_keys=False).apply(add_driver_features)

    # Basic labels: is this lap a pit lap?
    laps["is_pit_lap"] = laps["PitInTime"].notna().astype(int)

    # Normalised lap number
    total_laps = laps["LapNumber"].max()
    laps["LapNumberNorm"] = laps["LapNumber"] / total_laps

    # Clean TrackStatus to string
    laps["TrackStatus"] = laps["TrackStatus"].astype(str)

    # Add race identifiers
    laps["Track"] = event
    laps["Year"] = year
    laps["RaceId"] = f"{year}-{event}"

    # Select a core set of columns to keep for now
    cols = [
        "Year", "Track", "RaceId",
        "Driver", "Team", "LapNumber", "LapNumberNorm",
        "LapTime_s",
        "Stint", "LapsSinceLastPit", "StopsSoFar",
        "Compound", "TyreLife", "Position", "TrackStatus",
        "is_pit_lap",
    ]

    # Some older sessions may have NaNs in TyreLife; fill from LapsSinceLastPit for now
    if "TyreLife" in laps.columns:
        laps["TyreLife"] = laps["TyreLife"].fillna(laps["LapsSinceLastPit"])
    else:
        laps["TyreLife"] = laps["LapsSinceLastPit"]

    df = laps[cols].dropna()
    print(f"  -> {len(df)} rows kept for {year} {event}")
    return df



In [4]:
from src.config import TRAIN_RACES, TEST_RACES, PROCESSED_DIR

all_dfs = []

for (year, event) in TRAIN_RACES + TEST_RACES:
    df_race = make_lap_dataset_for_session(year, event, session_type="R")
    all_dfs.append(df_race)

full_dataset = pd.concat(all_dfs, ignore_index=True)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "all_laps_with_laptime.csv"
full_dataset.to_csv(out_path, index=False)

print("Full dataset shape:", full_dataset.shape)
print("Saved dataset to:", out_path)



Loading 2022 Bahrain R...


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', 

KeyError: "['LapTime_s'] not in index"